## Rag From Scratch: Query Transformations/Transilation

![alt text](<../../assets/Full Rag Pipeline.png>)

### Environment

In [12]:
import warnings
import os 
from dotenv import load_dotenv

# 0. Disable Warnings
warnings.filterwarnings("ignore")

# 1. Add parentheses to actually run the function
load_dotenv()

try: 
    # 2. Use .get() with a default empty string "" to avoid NoneType errors
    os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING_V2")
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")
    os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
    os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
    os.environ["USER_AGENT"] = "MyLangChainApp/1.0" # For WebBaseLoader
    print("Environment variables set successfully")
except Exception as e: 
    print(f"Error: {e}")

Environment variables set successfully


### Part 5: Multi Query

Flow:

![alt text](../../assets/MultiQuery.png)

#### Indexing

In [13]:
import bs4
# from langchain.text_splitter import RecursiveCharacterTextSplitter # Old import
from langchain_text_splitters import RecursiveCharacterTextSplitter # New import
from langchain_community.document_loaders import WebBaseLoader # New import
from langchain_community.vectorstores import Chroma # New import
from langchain_core.output_parsers import StrOutputParser # New import
from langchain_core.runnables import RunnablePassthrough # New import
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings # New import
from pathlib import Path
from langsmith import Client
client = Client()

# .parent.parent moves up twice to reach the root (rag_tutorial)
ROOT_DIR = Path.cwd().parents[1]

# Define the exact folder name at the root level
DB_DIR = ROOT_DIR / "db_blog"

In [ ]:
# #### INDEXING ####

# Load blog
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

# Split 
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50,
)

# Make Splits
splitted_docs = splitter.split_documents(blog_docs)


# Index
# There is no need to re-store these in the vector store, so we just gonna use the one we have.
embeddings = MistralAIEmbeddings(model = "mistral-embed")
vectorstore = Chroma.from_documents(
    documents = splitted_docs,
    embedding = embeddings,
    collection_name = "blog_posts", 
    persist_directory = str(DB_DIR)
)
retriever = vectorstore.as_retriever(search_kwargs = {"k":3}) 
# the number of the search_kwargs which is the number of most relevant document to retrieve 
# this number controls the documents in the langSmith trace view

#### Prompt

In [35]:
from langchain_core.prompts import ChatPromptTemplate

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Do not include any preamble.
Original question: {question}"""

# Prompt Template
prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_query_chain = (
    prompt_perspectives 
    | ChatMistralAI(model = "mistral-small-latest", temperature=0)
    | StrOutputParser()
    | (lambda x: [q for q in x.split("\n") if q.strip()])
)

# x going to split the query into multiple queries as a list
# the q it self doing a clean for each line in the list item by item.x``

In [36]:
from langchain_core.load import loads, dumps

def get_unique_union(documents: list[list]): 
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flatten_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flatten_docs))
    # Return list of Document objects
    return [loads(docs) for docs in unique_docs]


# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = (
    generate_query_chain | retriever.map() | get_unique_union)

docs = retrieval_chain.invoke({"question": question})

len(docs)



2

In [37]:
# Limit to the first 3 documents to keep your terminal output clean
for i, doc in enumerate(docs):
    print(f"\n{'='*40}")
    print(f"📄 DOCUMENT {i+1}")
    print(f"{'='*40}")
    
    # Safely extract the source URL from the metadata dictionary
    # We use .get() so it doesn't crash if the 'source' key is missing
    source = doc.metadata.get('source', 'Unknown Source')
    print(f"🔗 Source: {source}\n")
    
    # Print the first 400 characters of the page content to verify the text chunks
    print("📝 Content Preview:")
    print(f"{doc.page_content[:400]}...")


📄 DOCUMENT 1
🔗 Source: https://lilianweng.github.io/posts/2023-06-23-agent/

📝 Content Preview:
LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays an...

📄 DOCUMENT 2
🔗 Source: https://lilianweng.github.io/posts/2023-06-23-agent/

📝 Content Preview:
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller..

In [21]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_mistralai import ChatMistralAI

# RAG
template = """Answer the following question based on this context: 

Context: {context}


Question: {question}
""" 

prompt = ChatPromptTemplate.from_template(template)

llm = ChatMistralAI(model = "mistral-small-latest", temperature = 0)

final_rag_chain = (
    {"context": retrieval_chain, 
    "question": itemgetter("question")} 
    | prompt 
    | llm
    | StrOutputParser()
)

docs = final_rag_chain.invoke({"question": "What is the main topic of the blog post?"})

### Part 6: Rag Fusion

Flow:

![alt text](<../../assets/Rag Fusion.png>)